# Multimodal Late Fusion (Tongue + Voice)

This notebook performs late fusion between:
- Image model: `resnet50_best.pt`
- Voice model: `vocadiab_voice_svm_model.joblib`

It predicts diabetes risk by combining both modality probabilities.


In [19]:
!pip install -q timm librosa soundfile joblib

In [20]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import joblib

from sklearn.exceptions import InconsistentVersionWarning
import warnings
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)


In [21]:
# -----------------------------
# 1) Kaggle paths (auto-discovery)
# -----------------------------
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

resnet_matches = list(KAGGLE_INPUT.rglob("resnet50_best.pt"))
voice_matches = list(KAGGLE_INPUT.rglob("vocadiab_voice_svm_model.joblib"))

if not resnet_matches:
    raise FileNotFoundError("Could not find 'resnet50_best.pt' under /kaggle/input")
if not voice_matches:
    raise FileNotFoundError("Could not find 'vocadiab_voice_svm_model.joblib' under /kaggle/input")

RESNET_PATH = str(resnet_matches[0])
VOICE_MODEL_PATH = str(voice_matches[0])

print("RESNET_PATH:", RESNET_PATH)
print("VOICE_MODEL_PATH:", VOICE_MODEL_PATH)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


RESNET_PATH: /kaggle/input/models/edamhakim/t2d-tongue-detection/pytorch/default/1/resnet50_best.pt
VOICE_MODEL_PATH: /kaggle/input/models/edamhakim/vocadiab-voice-svm-model/scikitlearn/default/1/vocadiab_voice_svm_model.joblib


'cuda'

In [22]:
# -----------------------------
# 2) Load tongue model (ResNet50)
# -----------------------------
def build_tongue_model(checkpoint_path: str, device: str = "cpu"):
    model = models.resnet50(weights=None)
    # Matches checkpoint keys like fc.1.weight / fc.1.bias
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.fc.in_features, 1)
    )

    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()
    return model

tongue_model = build_tongue_model(RESNET_PATH, DEVICE)
print("Tongue model loaded.")


Tongue model loaded.


In [23]:
# -----------------------------
# 3) Load voice SVM pipeline
# -----------------------------
voice_bundle = joblib.load(VOICE_MODEL_PATH)
voice_pipeline = voice_bundle["pipeline"] if isinstance(voice_bundle, dict) else voice_bundle

print("Voice pipeline loaded:", type(voice_pipeline))
if hasattr(voice_pipeline, "named_steps") and "sc" in voice_pipeline.named_steps:
    print("Expected voice feature size:", voice_pipeline.named_steps["sc"].n_features_in_)


Voice pipeline loaded: <class 'sklearn.pipeline.Pipeline'>
Expected voice feature size: 2048


In [24]:
# -----------------------------
# 4) Preprocessing + modality predictors
# -----------------------------
IMG_SIZE = 224
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

img_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

@torch.no_grad()
def predict_tongue_proba(image_path: str) -> float:
    img = Image.open(image_path).convert("RGB")
    x = img_tfms(img).unsqueeze(0).to(DEVICE)
    logit = tongue_model(x).squeeze(0).squeeze(-1)
    p = torch.sigmoid(logit).item()
    return float(p)

def predict_voice_proba(feature_vec_2048: np.ndarray) -> float:
    """
    feature_vec_2048: shape (2048,) extracted from your voice pipeline.
    The stored sklearn pipeline handles scaling/PCA/SVM internally.
    """
    x = np.asarray(feature_vec_2048, dtype=np.float32).reshape(1, -1)
    if x.shape[1] != 2048:
        raise ValueError(f"Expected 2048 voice features, got {x.shape[1]}")

    if hasattr(voice_pipeline, "predict_proba"):
        p = voice_pipeline.predict_proba(x)[0, 1]
    else:
        # Fallback when no calibrated proba is available
        score = voice_pipeline.decision_function(x)[0]
        p = 1.0 / (1.0 + np.exp(-score))
    return float(p)


In [25]:
# -----------------------------
# 5) Late fusion
# -----------------------------
def late_fusion(
    p_tongue: float,
    p_voice: float,
    w_tongue: float = 0.5,
    w_voice: float = 0.5,
    threshold: float = 0.5
):
    s = w_tongue + w_voice
    if s <= 0:
        raise ValueError("Fusion weights must sum to > 0")

    w_tongue, w_voice = w_tongue / s, w_voice / s
    p_fused = (w_tongue * p_tongue) + (w_voice * p_voice)
    y_pred = int(p_fused >= threshold)

    return {
        "p_tongue": float(p_tongue),
        "p_voice": float(p_voice),
        "p_fused": float(p_fused),
        "prediction": y_pred,
        "label": "Diabetic" if y_pred == 1 else "Non-diabetic",
    }


In [26]:
# -----------------------------
# 6) Single-sample inference example
# -----------------------------
# Provide:
# - image_path: a tongue image
# - voice_features_path: .npy file with shape (2048,)

image_path = "/kaggle/input/your-data/sample_tongue.jpg"
voice_features_path = "/kaggle/input/your-data/sample_voice_features.npy"

# Uncomment to run once paths are valid:
# p_t = predict_tongue_proba(image_path)
# voice_vec = np.load(voice_features_path)
# p_v = predict_voice_proba(voice_vec)
# out = late_fusion(p_t, p_v, w_tongue=0.6, w_voice=0.4, threshold=0.5)
# print(json.dumps(out, indent=2))


In [27]:
# -----------------------------
# 7) Batch inference template
# -----------------------------
# CSV schema expected:
# image_path, voice_feat_path

def run_batch(csv_path: str, w_tongue: float = 0.5, w_voice: float = 0.5, threshold: float = 0.5):
    df = pd.read_csv(csv_path)
    rows = []

    for _, r in df.iterrows():
        p_t = predict_tongue_proba(r["image_path"])
        voice_vec = np.load(r["voice_feat_path"])
        p_v = predict_voice_proba(voice_vec)
        fused = late_fusion(p_t, p_v, w_tongue=w_tongue, w_voice=w_voice, threshold=threshold)

        rows.append({
            "image_path": r["image_path"],
            "voice_feat_path": r["voice_feat_path"],
            **fused
        })

    return pd.DataFrame(rows)

# Example:
# results_df = run_batch("/kaggle/input/your-data/fusion_manifest.csv", w_tongue=0.6, w_voice=0.4)
# results_df.head()


## Notes

- This notebook assumes your voice branch already outputs a 2048-dimensional feature vector per sample.
- If your voice preprocessing starts from raw audio (`.wav`), plug that feature extractor before `predict_voice_proba`.
- Tune `w_tongue`, `w_voice`, and the fusion threshold on a validation set for best performance.
